# GBM calibration & Monte Carlo — period 2018–2019

**Period file:** **2018-01-01 → 2019-12-31**.

| Role | Ticker |
|------|--------|
| Primary | **SPY** |
| Secondary | AAPL |
| Secondary | MSFT |

**Section roles**
- **§4 Calibration only:** choose lookback / rolling, **Reestimate**, inspect estimated parameters (no Monte Carlo plots here).
- **§5 Monte Carlo only:** Start / Restart; simulated paths and history comparison.

True rolling rule: at each update, re-estimate \(\hat\mu,\hat\sigma\) from the current window and use them for the next MC segment. See `ROLLING_CALIBRATION.md`.


## 0. Setup


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

DATA = Path("..") / "data"
PERIOD_START = pd.Timestamp("2018-01-01")
PERIOD_END = pd.Timestamp("2019-12-31")
TICKERS = ["AAPL", "MSFT", "SPY"]
N_DAYS = 252
COLORS = {"AAPL": "#1f77b4", "MSFT": "#ff7f0e", "SPY": "#2ca02c"}

WINDOW_OPTIONS = {
    "3 months": pd.DateOffset(months=3),
    "6 months": pd.DateOffset(months=6),
    "1 year": pd.DateOffset(years=1),
    "2 years": pd.DateOffset(years=2),
    "5 years": pd.DateOffset(years=5),
}
ROLLING_OPTIONS = ["daily", "monthly", "none"]

prices = pd.read_csv(DATA / "equity" / "prices_clean.csv", parse_dates=["Date"]).set_index("Date").sort_index()
period_prices = prices.loc[PERIOD_START:PERIOD_END, TICKERS].copy()
log_returns_all = np.log(prices[TICKERS]).diff()

rolling = {}
cal_meta = {}

print(f"Price sample: {prices.index.min().date()} → {prices.index.max().date()}")
print(
    f"Period rows: {len(period_prices)} trading days "
    f"({period_prices.index.min().date()} → {period_prices.index.max().date()})"
)
period_prices.head()

# --- clean plotting / widget memory (important after reopen) ---
plt.close("all")
plt.ioff()


## 1. Stock price trends (2018–2019)

Adjusted close for AAPL, MSFT, and SPY (primary).


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
for ax, ticker in zip(axes, TICKERS):
    s = period_prices[ticker].dropna()
    ax.plot(s.index, s.values, color=COLORS[ticker], lw=1.4)
    role = "primary" if ticker == "SPY" else "secondary"
    ax.set_ylabel("Adj close")
    ax.set_title(f"{ticker} ({role}) — adjusted close, 2018–2019")
axes[-1].set_xlabel("Date")
fig.suptitle("Stock price trends — period 2018–2019", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
display(period_prices.describe().T[["count", "mean", "min", "max"]].round(4))


## 2. Strike prices in this period

Unique strikes \(K\) from `*_options_panel.csv` with `trading_date` in **2018-01-01 → 2019-12-31**.

> AAPL strikes are on the option/contract scale; equity adj closes are split-adjusted.


In [ ]:
for ticker in TICKERS:
    path = DATA / "options" / "processed" / f"{ticker}_options_panel.csv"
    opt = pd.read_csv(path, usecols=["trading_date", "K"], parse_dates=["trading_date"])
    m = (opt["trading_date"] >= PERIOD_START) & (opt["trading_date"] <= PERIOD_END)
    sub = opt.loc[m]
    uniq = np.sort(sub["K"].dropna().unique())
    dmin, dmax = sub["trading_date"].min(), sub["trading_date"].max()
    display(Markdown(
        f"### {ticker} — {len(uniq)} unique strikes "
        f"(options quotes {dmin.date() if pd.notna(dmin) else 'n/a'} → "
        f"{dmax.date() if pd.notna(dmax) else 'n/a'})"
    ))
    print("Strikes K:", ", ".join(f"{x:g}" for x in uniq))
    display(pd.DataFrame({"K": uniq}).T)


## 3. Estimation formulas (GBM)

$$dS_t = \mu S_t\, dt + \sigma S_t\, dW_t$$

$$S_{t+\Delta t} = S_t \exp\Big(\big(\mu - \tfrac{1}{2}\sigma^2\big)\Delta t + \sigma\sqrt{\Delta t}\, Z\Big),\quad Z\sim N(0,1)$$

| Parameter | Estimator (\(N=252\)) |
|-----------|------------------------|
| \(\hat\mu\) | \(\bar r \times 252\) |
| \(\hat\sigma\) | \(s_r \times \sqrt{252}\) |
| \(r_t\) | \(\ln(S_t/S_{t-1})\) |

**True rolling:** at each update date, re-estimate from the lookback window ending there; those params drive the next Monte Carlo segment.


## 4. Calibration only — 2018–2019

Sliders + **Reestimate**. Shows **only** the rolling parameter graphs (no tables).  
Monte Carlo vs history is in **§5** only — one pair per company.


In [ ]:
def estimate_mu_sigma(log_rets: pd.Series):
    x = log_rets.dropna()
    n = int(x.shape[0])
    if n < 2:
        return np.nan, np.nan, n
    return float(x.mean() * N_DAYS), float(x.std(ddof=1) * np.sqrt(N_DAYS)), n


def _slice_window(rets: pd.Series, end: pd.Timestamp, offset: pd.DateOffset) -> pd.Series:
    start = end - offset
    return rets.loc[(rets.index > start) & (rets.index <= end)]


def calibrate_ticker(ticker: str, window_label: str, rolling_mode: str) -> pd.DataFrame:
    rets = log_returns_all[ticker].dropna()
    offset = WINDOW_OPTIONS[window_label]
    rows = []

    if rolling_mode == "daily":
        update_dates = rets.loc[(rets.index >= PERIOD_START) & (rets.index <= PERIOD_END)].index
    elif rolling_mode == "monthly":
        t0 = period_prices[ticker].dropna().index[0]
        month_ends = pd.date_range(PERIOD_START, PERIOD_END, freq="ME")
        update_dates = pd.DatetimeIndex([t0]).append(month_ends).unique().sort_values()
    else:
        update_dates = pd.DatetimeIndex([period_prices[ticker].dropna().index[0]])

    for t_u in update_dates:
        window = _slice_window(rets, pd.Timestamp(t_u), offset)
        mu, sigma, n = estimate_mu_sigma(window)
        if n < 2:
            continue
        rows.append({
            "date": pd.Timestamp(t_u),
            "window_start": window.index.min(),
            "window_end": window.index.max(),
            "n_days": n,
            "mu": mu,
            "sigma": sigma,
        })
    return pd.DataFrame(rows)


def param_schedule_for_steps(ticker: str, cal_table: pd.DataFrame):
    hist = period_prices[ticker].dropna()
    dates = hist.index
    n_steps = len(dates) - 1
    cal = cal_table.sort_values("date").reset_index(drop=True)
    cal_dates = pd.to_datetime(cal["date"]).to_numpy()
    mu_arr = cal["mu"].to_numpy(dtype=float)
    sig_arr = cal["sigma"].to_numpy(dtype=float)
    mu_step = np.empty(n_steps, dtype=float)
    sig_step = np.empty(n_steps, dtype=float)
    for i in range(n_steps):
        idx = np.searchsorted(cal_dates, np.datetime64(dates[i]), side="right") - 1
        if idx < 0:
            idx = 0
        mu_step[i] = mu_arr[idx]
        sig_step[i] = sig_arr[idx]
    return dates, mu_step, sig_step, float(hist.iloc[0]), hist


def _show_fig(fig):
    """Show a figure exactly once as PNG.

    Root cause of duplicates: with %matplotlib inline, display(fig) inside an
    Output can ALSO be flushed again by the inline backend → same graph twice
    (worse after reopen when the old kernel is still alive). Saving PNG bytes
    and closing the Figure first avoids that second paint.
    """
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def plot_rolling_paths(rolling_dict: dict, window_label: str, rolling_mode: str):
    """One rolling-parameter figure (graphs only)."""
    with plt.ioff():
        fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
        for t in TICKERS:
            r = rolling_dict[t]
            x = pd.to_datetime(r["date"])
            mark = "o" if len(r) < 40 else None
            axes[0].plot(x, r["mu"], lw=1.2, label=t, color=COLORS[t], marker=mark, ms=3)
            axes[1].plot(x, r["sigma"], lw=1.2, label=t, color=COLORS[t], marker=mark, ms=3)
        axes[0].axhline(0, color="0.5", lw=0.8)
        axes[0].set_ylabel("μ̂ (annual)")
        axes[0].set_title(f"Estimated drift — {rolling_mode}, lookback {window_label}")
        axes[0].legend(frameon=False, ncol=3)
        axes[1].set_ylabel("σ̂ (annual)")
        axes[1].set_title(f"Estimated volatility — {rolling_mode}, lookback {window_label}")
        axes[1].set_xlabel("Date")
        axes[1].legend(frameon=False, ncol=3)
        fig.tight_layout()
    _show_fig(fig)


# Reset kernel-side UI handles so reopen + Run All cannot reuse stale widgets
plt.close("all")
plt.ioff()
rolling = {}
cal_meta = {}

cal_out = widgets.Output(layout=widgets.Layout(width="100%"))
window_slider = widgets.SelectionSlider(
    options=list(WINDOW_OPTIONS.keys()),
    value="3 months",
    description="Lookback",
    continuous_update=False,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="420px"),
)
rolling_slider = widgets.SelectionSlider(
    options=ROLLING_OPTIONS,
    value="daily",
    description="Rolling",
    continuous_update=False,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="420px"),
)
btn_reestimate = widgets.Button(description="Reestimate", button_style="primary", icon="refresh")

cal_ui = widgets.VBox([
    widgets.HTML(
        "<b>§4 Calibration (graphs only)</b> — lookback + rolling, then <b>Reestimate</b>. "
        "No parameter tables. No Monte Carlo here."
    ),
    window_slider,
    rolling_slider,
    btn_reestimate,
    cal_out,
])


def reestimate(_=None):
    global rolling, cal_meta
    window_label = window_slider.value
    rolling_mode = rolling_slider.value
    rolling = {t: calibrate_ticker(t, window_label, rolling_mode) for t in TICKERS}
    cal_meta = {"window_label": window_label, "rolling_mode": rolling_mode}

    with cal_out:
        clear_output(wait=True)
        display(Markdown(
            f"**Calibration updated:** lookback=`{window_label}`, rolling=`{rolling_mode}` "
            f"(n_updates: " + ", ".join(f"{t}={len(rolling[t])}" for t in TICKERS) + ")"
        ))
        plot_rolling_paths(rolling, window_label, rolling_mode)
        display(Markdown("Go to **§5** and click **Start** for one MC pair per company."))


btn_reestimate.on_click(reestimate)
display(cal_ui)
reestimate()


## 5. Monte Carlo only — one graph pair per company (2018–2019)

| Left | Right |
|------|--------|
| Monte Carlo paths + expected path | Expected path vs historical prices |

Uses latest **Reestimate** from §4. **Start** / **Restart** redraw that single pair (never stacks another copy).


In [ ]:
def simulate_gbm_rolling(mu_step, sigma_step, S0, n_paths, seed):
    rng = np.random.default_rng(seed)
    n_steps = len(mu_step)
    dt = 1.0 / N_DAYS
    paths = np.empty((n_paths, n_steps + 1), dtype=float)
    paths[:, 0] = S0
    for i in range(n_steps):
        mu, sigma = mu_step[i], sigma_step[i]
        z = rng.standard_normal(n_paths)
        paths[:, i + 1] = paths[:, i] * np.exp(
            (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z
        )
    return paths


def _show_fig(fig):
    """Show a figure exactly once as PNG.

    Root cause of duplicates: with %matplotlib inline, display(fig) inside an
    Output can ALSO be flushed again by the inline backend → same graph twice
    (worse after reopen when the old kernel is still alive). Saving PNG bytes
    and closing the Figure first avoids that second paint.
    """
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def _draw_ticker_pair(ticker: str, out: widgets.Output, seed: int, n_paths: int = 80):
    """Replace contents of `out` with exactly one 1×2 figure."""
    with out:
        clear_output(wait=True)
        if ticker not in rolling or len(rolling[ticker]) == 0:
            display(Markdown("Run **Reestimate** in §4 first."))
            return
        dates_now, mu_now, sig_now, S0_now, hist_now = param_schedule_for_steps(
            ticker, rolling[ticker]
        )
        paths = simulate_gbm_rolling(mu_now, sig_now, S0_now, n_paths, seed)
        expected = paths.mean(axis=0)

        with plt.ioff():
            fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
            axes[0].plot(dates_now, paths.T, color=COLORS[ticker], alpha=0.12, lw=0.7)
            axes[0].plot(dates_now, expected, color="black", lw=2.2, label="expected path (MC mean)")
            axes[0].set_title(f"{ticker}: Monte Carlo")
            axes[0].set_ylabel("price")
            axes[0].legend(loc="best", frameon=False)

            axes[1].plot(dates_now, hist_now.values, color=COLORS[ticker], lw=1.8, label="historical")
            axes[1].plot(dates_now, expected, color="black", lw=2.0, ls="--", label="expected path")
            axes[1].set_title(f"{ticker}: expected vs history")
            axes[1].set_ylabel("price")
            axes[1].legend(loc="best", frameon=False)
            for ax in axes:
                ax.set_xlabel("date")
            fig.suptitle(
                f"{ticker} | seed={seed} | {cal_meta.get('rolling_mode')} / {cal_meta.get('window_label')}",
                fontsize=11,
                y=1.02,
            )
            fig.tight_layout()
        _show_fig(fig)
        rmse = float(np.sqrt(np.mean((expected - hist_now.values) ** 2)))
        print(f"RMSE(expected vs historical) = {rmse:.4f} | seed = {seed}")


def make_ticker_panel(ticker: str, n_paths: int = 80):
    """One Output per company. Start/Restart only replace that Output (no stacking)."""
    state = {"seed": 42}
    mode = cal_meta.get("rolling_mode", "?")
    win = cal_meta.get("window_label", "?")
    out = widgets.Output(layout=widgets.Layout(width="100%"))
    btn_start = widgets.Button(description="Start", button_style="success", icon="play")
    btn_restart = widgets.Button(description="Restart", button_style="warning", icon="refresh")
    info = widgets.HTML(f"<b>{ticker}</b> — one graph pair | lookback={win}, mode={mode}")

    busy = {"on": False}

    def on_start(_):
        if busy["on"]:
            return
        busy["on"] = True
        try:
            _draw_ticker_pair(ticker, out, state["seed"], n_paths)
        finally:
            busy["on"] = False

    def on_restart(_):
        if busy["on"]:
            return
        busy["on"] = True
        try:
            state["seed"] = int(np.random.default_rng().integers(0, 1_000_000_000))
            _draw_ticker_pair(ticker, out, state["seed"], n_paths)
        finally:
            busy["on"] = False

    btn_start.on_click(on_start)
    btn_restart.on_click(on_restart)
    return widgets.VBox([info, widgets.HBox([btn_start, btn_restart]), out])


plt.close("all")
plt.ioff()

mc_host = widgets.VBox([])
children = [widgets.HTML("<b>§5 Monte Carlo — click <i>Start</i> once per company (one pair only)</b>")]
for ticker in TICKERS:
    role = "primary" if ticker == "SPY" else "secondary"
    children.append(widgets.HTML(f"<h4 style='margin:8px 0 4px'>{ticker} ({role})</h4>"))
    children.append(make_ticker_panel(ticker))
mc_host.children = tuple(children)
display(mc_host)


## 6. Reminder

1. **§4:** sliders → **Reestimate** → read parameter tables / rolling μ̂,σ̂ charts.  
2. **§5:** **Start** → Monte Carlo + expected vs history (one pair per ticker).  
3. **Restart** only changes the random seed.
